In [ ]:
# Cell 1: entorno
import torch, shutil
print("cuda:", torch.cuda.is_available())


In [ ]:
# Cell 2: binario estatico de GNINA 1.3.3 (x86_64, CUDA 12.8) + cuDNN 9 por pip
!wget -q -O /tmp/gnina https://github.com/gnina/gnina/releases/download/v1.3.3/gnina.cuda12.8.static
!chmod +x /tmp/gnina
!ls -la /tmp/gnina
!pip -q install nvidia-cudnn-cu12 nvidia-cuda-runtime-cu12 nvidia-cublas-cu12 nvidia-cufft-cu12 nvidia-curand-cu12 nvidia-cusparse-cu12 nvidia-cusolver-cu12 nvidia-cuda-nvrtc-cu12 2>&1 | tail -1
import glob, os
nvdirs = (glob.glob('/usr/local/lib/python*/dist-packages/nvidia/*/lib')
           + glob.glob('/opt/conda/lib/python*/site-packages/nvidia/*/lib')
           + glob.glob('/usr/lib/python3*/dist-packages/nvidia/*/lib'))
assert nvdirs, 'no encontre librerias nvidia de pip'
os.environ['LD_LIBRARY_PATH'] = ':'.join(nvdirs) + ':' + os.environ.get('LD_LIBRARY_PATH', '')
print('LD_LIBRARY_PATH:', os.environ['LD_LIBRARY_PATH'])
!/tmp/gnina --version 2>&1 | tail -2


In [ ]:
# Cell 3: score_only de las 470 poses (sin docking); cada pdbqt trae 9 MODEL: uso el 1
import glob, os, re, csv, subprocess

rec = (glob.glob("/kaggle/input/*/receptor_tbk1.pdbqt") or glob.glob("/kaggle/input/**/receptor_tbk1.pdbqt", recursive=True))[0]
dir_poses = (glob.glob("/kaggle/input/*/out") or glob.glob("/kaggle/input/**/out", recursive=True))[0]
poses = sorted(glob.glob(os.path.join(dir_poses, "*.pdbqt")))
assert os.environ.get('LD_LIBRARY_PATH'), 'falta cuDNN: la celda 2 debe correr antes'
print("receptor:", rec, "| poses:", len(poses))

GNINA = "/tmp/gnina"
salida = "/kaggle/working/gnina_scores.csv"

def dividir_modelos(pose):
    """Parte el pdbqt de 9 modelos en ficheros de un modelo, SIN las lineas
    MODEL/ENDMDL: a smina/gnina en modo score le trastan (Parse error ENDMDL)."""
    txt = open(pose).read()
    if 'MODEL' not in txt:
        return [pose]
    modelos, actual = [], None
    for linea in txt.splitlines():
        if linea.startswith('MODEL'):
            actual = []
        elif linea.startswith('ENDMDL'):
            modelos.append(actual); actual = None
        elif actual is not None:
            actual.append(linea)
    rutas = []
    destino = '/kaggle/working/modelos'
    os.makedirs(destino, exist_ok=True)
    base = os.path.basename(pose)
    for i, m in enumerate(modelos[:1]):
        r = os.path.join(destino, base + '.m%d.pdbqt' % i)
        open(r, 'w').write('\n'.join(m) + '\n')
        rutas.append(r)
    return rutas

def puntuar(pose):
    r = subprocess.run([GNINA, "-r", rec, "-l", pose, "--score_only"],
                       capture_output=True, text=True, timeout=600)
    t = r.stdout + r.stderr
    aff = re.findall(r"Estimated[^=\n]*=\s*(-?[0-9]+\.[0-9]+)", t)
    cnn = re.findall(r"CNNscore[:=\s]+([0-9]+\.[0-9]+)", t)
    cnnaff = re.findall(r"CNNaffinity[:=\s]+(-?[0-9]+\.[0-9]+)", t)
    afinidades = [float(x) for x in aff]
    return (min(afinidades) if afinidades else None,
            float(cnn[0]) if cnn else None,
            float(cnnaff[0]) if cnnaff else None, t)

# diagnostico: la primera pose, cruda
m0 = dividir_modelos(poses[0])
a, s, ca, t = puntuar(m0[0])
open('/kaggle/working/ejemplo_salida.txt', 'w').write(t)
print('CRUDA pose1:', t[:600])
print('valores pose1:', a, s, ca)

filas = []
for i, p in enumerate(poses):
    nombre = os.path.basename(p)
    chembl = re.search(r"CHEMBL\d+", nombre).group(0)
    try:
        m = dividir_modelos(p)
        a, s, ca, _ = puntuar(m[0])
    except Exception as e:
        a = s = ca = None
        print("FALLO", nombre, repr(e)[:100])
    filas.append({"chembl_id": chembl, "vina_affinity": a,
                  "cnnscore": s, "cnnaffinity": ca})
    if (i + 1) % 25 == 0:
        print("%d/%d" % (i + 1, len(poses)))
        with open(salida, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=["chembl_id", "vina_affinity", "cnnscore", "cnnaffinity"])
            w.writeheader(); w.writerows(filas)

with open(salida, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["chembl_id", "vina_affinity", "cnnscore", "cnnaffinity"])
    w.writeheader(); w.writerows(filas)
print("csv escrito:", salida, len(filas), "filas")


In [ ]:
# Cell 4: comparacion con la verdad
import glob, pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

verdad = pd.read_csv((glob.glob("/kaggle/input/*/verdad_tbk1_v2.csv") or glob.glob("/kaggle/input/**/verdad_tbk1_v2.csv", recursive=True))[0]).rename(columns={"molecule_chembl_id": "chembl_id"})
scores = pd.read_csv("/kaggle/working/gnina_scores.csv")
m = verdad.merge(scores, on="chembl_id", how="inner")
print("cruce:", len(m), "de", len(verdad), "ligandos")

y = (m["clase"] == "activo").astype(int)
for col in ["vina_affinity", "cnnscore", "cnnaffinity"]:
    sub = m.dropna(subset=[col])
    if len(sub) < 10:
        print(col, ": pocas filas (%d)" % len(sub))
        continue
    ys = y.loc[sub.index]
    rho = spearmanr(sub[col], sub["pchembl_mediana"]).correlation
    print("%-14s AUROC %.3f | Spearman %.3f | n=%d" % (col, roc_auc_score(ys, sub[col]), rho, len(sub)))
m.to_csv("/kaggle/working/veredicto_gnina.csv", index=False)
print(m.head())
